# Architecture

The Python and R front ends parse formulas and expose model methods. The
PyO3 and extendr bindings convert their inputs to Rust types, and
`mboostrs-core` prepares learners and runs the fitting algorithms.


```{figure} ../figures/mermaid-3c2d995fa6eb.svg
:name: mermaid-3c2d995fa6eb
:alt: Sequence from user code through the front end and language binding to learner preparation and the iterative Rust boosting engine, followed by reconstruction of the fitted model.

Python and R front ends prepare learners, call the shared Rust engine, and reconstruct a fitted model from the returned boosting path.
```


```rust
macro_rules! with_concrete_family {
    ($name:expr, $params:expr, |$fam:ident| $body:expr) => {{
        match $name {
            "gaussian"  => { let mut $fam = mboostrs_core::Gaussian;        Ok($body) }
            "binomial"  => { let mut $fam = mboostrs_core::Binomial;        Ok($body) }
            "poisson"   => { let mut $fam = mboostrs_core::Poisson;         Ok($body) }
            "huber"     => { let mut $fam = mboostrs_core::Huber::new(...); Ok($body) }
            // ... 16 single-parameter families ...
            _ => Err(pyo3::exceptions::PyValueError::new_err(
                format!("unknown family: {}", $name))),
        }
    }};
}
```


Each arm creates a stack-allocated concrete family, so the inner boosting loop
monomorphizes and inlines `ngradient_and_risk` / `calibrate`. This eliminates
trait-object dispatch overhead in the hot path. The macro is used at every
entry point that takes a family string: `fit_glmboost` (`lib.rs:346`),
`fit_gamboost_specs`, the `cvrisk_*` entry points, and
`stabsel_*` (`lib.rs:3552`).

### Family name mapping

Python class names map to Rust names with some irregularity (`families.py:63-75`):

| Python class | Rust name |
|---|---|
| `Gaussian` | `gaussian` |
| `Binomial` | `binomial` |
| `Huber` | `huber` |
| `GammaReg` | **`gammasreg`** (extra 's') |
| `Quantile` | **`quantreg`** |
| `Expectile` | **`expectreg`** |

When adding a family or entry point, match the macro arm to the Python-side
family string exactly.

## Responsibility matrix

The following table locates formula handling, numerical calculations and model methods.

| Concern | Front-end | Binding | Core |
|---|---|---|---|
| Formula parsing | ✓ (`formula.py`) | — | — |
| Response validation (`len(w) == n`) | ✓ (raises `ValueError`) | — | — |
| Column extraction, centering means | ✓ | — | — |
| `{0,1}` $\rightarrow$ `{-1,+1}` recode | — | — | ✓ (`Family::encode_response`) |
| Family name string $\rightarrow$ concrete type | — | ✓ (`with_concrete_family!`) | — |
| Design matrix construction (`bbs`, `bspatial`, …) | — | — | ✓ (`learners.rs`) |
| Penalty matrix construction | — | — | ✓ (`learners.rs`) |
| `df` $\rightarrow$ `λ` solve | — | — | ✓ (`linalg.rs`) |
| Gram factorization (Cholesky) | — | — | ✓ (`linalg.rs`) |
| Negative gradient, risk, offset | — | — | ✓ (`families.rs`) |
| Per-step back-solve, selection, update | — | — | ✓ (`engine.rs`) |
| Risk path, fitted values, selected sequence | — | — | ✓ (`engine.rs`) |
| AIC, hatvalues (need stored basis) | — | ✓ (`learner_store`) | ✓ (compute) |
| `predict(newdata=...)` | ✓ (column extraction) | ✓ (`predict_newdata_grouped`) | ✓ (`BasisInfo::predict_into`) |
| `cvrisk` fold machinery | ✓ (`cvrisk.py`) | ✓ (parallel) | ✓ (`cv.rs`) |
| Plotting | ✓ (`plot.py`) | — | — |
| Pickling | ✓ (`__getstate__`) | — | — |

The core implements distributional calculations and numerical fitting. The
front ends provide formula syntax and model methods; the bindings convert
arrays and learner specifications between those layers.

## The Gaussian fast path

When the family is Gaussian, the risk is inbag, and every learner is a linear
smoother, the engine can bypass the $O(n)$ per-iteration gradient computation
entirely. This is `try_fit_gaussian_gram` (`engine.rs:654`), and it avoids an $O(n)$ gradient pass at each iteration. It is enabled by the
`Family::supports_gram_update()` opt-in (`families.rs:58`), which only
`Gaussian` returns `true` for. The mechanics (cross-Gram matrices, RHS updates,
$O(n + \sum p_w p_{j^\star})$ per step) are in [](engine.ipynb); the memory
scaling that gates it is in [](performance.ipynb).

The fast path returns the same `BoostingPath` as the general loop. If a
precondition fails, the engine uses the general loop. Gaussian parity tests
should cover both paths, with the fast path enabled by default and disabled
by adding a `btree` learner.

## Adding a new family

A new single-parameter family is a self-contained addition. The steps:

1. **Core: implement `Family`.** Add a struct in `families.rs` and `impl Family
   for YourFamily`. Provide `name`, `offset`, `negative_gradient`, `risk`,
   `response`, and (if there is a nuisance parameter) `calibrate` / `nuisance`
   / `set_nuisance`. Override `ngradient_and_risk` if you can share link
   evaluations between the gradient and the risk.
2. **Core: add a finite-difference test.** Add a `#[test]` that calls the
   finite-difference gradient verifier (see [](testing.ipynb)) on your family.
   This catches sign errors and chain-rule mistakes before they reach parity.
3. **Binding: add a macro arm.** Add `"yourname" => { let mut fam = YourFamily::new(...); Ok($body) }`
   to `with_concrete_family!` in `crates/mboostrs-py/src/lib.rs`, and the
   analogous arm to the extendr module in `crates/mboostrs-r/src/lib.rs`.
4. **Front-end: add a Python class.** Add `YourFamily` to `families.py` mapping
   to the Rust name string (mind the suffix irregularities). Add the R
   constructor to `mboostrs/R/mboostrs.R` and an export line to `NAMESPACE`.
5. **Tests: add a parity test.** Create
   `tests/test_glmboost_parity/test_yourfamily_bols.py` following the template
   in [](testing.ipynb). Set the tolerance from observed divergence against R and
   document the source of any gap.

An LSS family follows the same process through the `LssFamily` trait; see the
"Adding an LSS family" table in [](lss_engine.ipynb).

### Compiled families and run-time callbacks

`mboost::Family()` builds a boosting family from R closures at run time, and
`gamlss.dist` has about 116 distributions reachable that way. This package has
no equivalent: adding a family requires compiling Rust code. The following
measurements describe the costs a run-time callback would need to address.

**The gradient is evaluated once per iteration, per fold.** A 1,000-iteration
fit calls it 1,000 times; `cvrisk` with 10 folds calls it 10,000; stability
selection with 50 complementary pairs calls it 100,000. Each call would cross
the language boundary with the full residual vector.

**Gradient evaluation cost.** Measured on this machine, 1,000 NumPy gradient evaluations over 50,000 rows take 26 ms against
102 ms for the whole 1,000-iteration fit — about a quarter, before counting
the FFI itself.

**Parallel selection.** The current bindings release the GIL while fitting,
and the Rust engine can evaluate block learners in parallel. On 24 `bbs`
learners over 20,000 rows, the parallel block route takes **136 ms** against
**259 ms** pinned to one thread, a 1.9x difference. These timings compare
thread settings for compiled families; they do not measure a callback
implementation. A callback interface would need to reacquire the GIL for
Python calls and release it for the native candidate scan.

**Callback validation.** A callback can raise an exception, return a vector
with the wrong length or non-finite entries, or mutate its input. Supporting
these callbacks inside the fitting loop would require error propagation and
validation of each returned gradient, including another pass over its entries.

The PIG family illustrates the compiled-family workflow for a distribution
with more involved calculations: an exact log-likelihood needing a special function, a gradient in terms
of a Bessel ratio, and four acceptance gates before fitting was enabled. See
`mod pig_contract` in `crates/mboostrs-core/src/lss.rs`.

For a distribution unavailable here, R's `gamlss` provides additional families;
see [](../usage/r_migration.ipynb).

## Adding a new base learner

A new base learner is a new `PreparedLearner` variant. The steps:

1. **Core: add the variant.** Add `YourLearner { ... }` to the
   `PreparedLearner` enum in `learners.rs`, holding the design, the penalty,
   the cached factorization (or split state for `btree`-like learners), and
   the scratch buffers.
2. **Core: add a `prepare_*` function.** Build the design matrix, build the
   penalty, factorize the Gram, and construct the variant. This function calls
   `df_to_lambda` and receives per-learner data from the binding.
3. **Core: implement `fit_step_ref`, `predict_into`, `reweight`.** The engine calls
   these three methods to fit, predict and rebuild under new weights. For a linear smoother, `fit_step_ref` is a
   `GramSolver` back-solve; for a non-linear learner like `btree`, it is a
   search.
4. **Binding: extend the learner schema.** Add one tagged decoder arm to
   `parse_learner_blueprint_list` in `crates/mboostrs-py/src/lib.rs` and the
   corresponding R decoder. Production fit, CV, stability selection, and
   prediction then share that self-describing learner record. Do not add a
   new parallel array to the deprecated wide compatibility boundary.
5. **Front-end: parse the formula term.** Add a regex and a builder to
   `formula.py`. If the learner takes a matrix argument (`bmrf`'s `K`,
   `bconstrained`'s `C`), emit it with the matrix set to `None` and document
   that the caller must fill it in.
6. **Front-end: preserve the replay recipe.** Lower the learner to one spec
   in formula order and ensure `LearnerStore` can rebuild its design for
   new-data prediction and resampling.
7. **Tests: add unit + parity tests.** Pin the design dimensions, the penalty
   structure, and the parity tolerance against R.

The engine calls `fit_step_ref`, `predict_into` and `reweight` through
`PreparedLearner`; each variant supplies its own implementation.

## See also

- [](engine.ipynb) — the inner loop, the Gaussian fast path, and a worked-example
  trace of the first iteration through the engine buffers.
- [](base_learners.ipynb) — per-variant details for every `PreparedLearner`.
- [](linear_algebra.ipynb) — the `GramSolver` and the `df_to_lambda` solve.
- [](frontends.ipynb) — how the front ends use the path returned by the core.
- [](testing.ipynb) — how parity and finite-difference tests protect the
  family and learner methods above.
- [](../methodology/functional_gradient_boosting) — the conceptual algorithm
  this architecture implements (offset, gradient, selection, update).
- [](../usage/glmboost) and [](../usage/gamboost) — runnable examples
  that exercise every component in this chapter.